In [ ]:
from dotenv import load_dotenv

from langchain import hub
from langchain_teddynote import logging
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_openai import ChatOpenAI
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.runnables import chain
from langchain_teddynote.messages import stream_response
from langchain_teddynote.callbacks import StreamingCallback
from langchain_core.output_parsers import StrOutputParser

In [ ]:
load_dotenv()

In [ ]:
logging.langsmith("langchain-summary-maprefine")

# Map-Refine

## Map

In [ ]:
llm_3 = ChatOpenAI(
    temperature=0, 
    model_name="gpt-4o-mini"
)

In [ ]:
map_summary = hub.pull("teddynote/map-summary-prompt")

In [ ]:
map_summary.pretty_print()

In [ ]:
map_chain = map_summary | llm | StrOutputParser()

In [ ]:
print(map_chain.invoke({"documents": docs[0], "language": "Korean"}))

In [ ]:
input_doc = [{"documents": doc, "language": "Korean"} for doc in docs]

In [ ]:
input_doc

In [ ]:
print(map_chain.batch(input_doc))  # 모든 문서에 대한 요약본

## Refine

In [ ]:
refine_prompt = hub.pull("teddynote/refine-prompt")

In [ ]:
refine_prompt.pretty_print()

In [ ]:
refine_llm = ChatOpenAI(
    temperature=0,
    model_name="gpt-4o-mini",
)

In [ ]:
refine_chain = refine_prompt | refine_llm | StrOutputParser()

In [ ]:
@chain
def map_refine_chain(docs):

    # map chain 생성
    map_summary = hub.pull("teddynote/map-summary-prompt")

    map_chain = (
        map_summary
        | ChatOpenAI(
            model_name="gpt-4o-mini",
            temperature=0,
        )
        | StrOutputParser()
    )

    input_doc = [{"documents": doc.page_content, "language": "Korean"} for doc in docs]

    # 첫 번째 프롬프트, ChatOpenAI, 문자열 출력 파서를 연결하여 체인을 생성합니다.
    doc_summaries = map_chain.batch(input_doc)

    refine_prompt = hub.pull("teddynote/refine-prompt")

    refine_llm = ChatOpenAI(
        model_name="gpt-4o-mini",
        temperature=0,
        callbacks=[StreamingCallback()],
        streaming=True,
    )

    refine_chain = refine_prompt | refine_llm | StrOutputParser()

    previous_summary = doc_summaries[0]

    for current_summary in doc_summaries[1:]:

        previous_summary = refine_chain.invoke(
            {
                "previous_summary": previous_summary,
                "current_summary": current_summary,
                "language": "Korean",
            }
        )
        print("\n\n-----------------\n\n")

    return previous_summary